# 04 — T1 interpretability + ablations
**Project:** MAG2D-NC | **Phase:** F5 / Step 3 | **Protocol:** v1.1 (frozen)

T1 passed the frozen success criterion (F1-macro 0.615 vs null 0.49,
perm p=0.003, delta=0.69). This notebook delivers the C2 evidence and two
frozen ablation axes:

1. **Interpretability** — SHAP (LightGBM) + standardized coefficients (logreg),
   collected strictly on out-of-fold test sets inside the same grouped CV;
   rank-stability across seeds (mean pairwise Spearman).
2. **Confusion analysis** — pooled OOF confusion matrix + fine-class resolution
   (which label4 classes drive binary errors; DM_SS recall in particular).
3. **Ablation A1** — descriptor-group removal (−comp / −soc / −sym), full
   nested search, same splits/seeds/budget.
4. **Ablation A3** — learning curve (25/50/75/100% of training GROUPS),
   modal HPs fixed (disclosed; search at every fraction is prohibitive).

Self-contained: reruns its own nested search (does not depend on notebook 03
memory). Runtime ~30-50 min full; QUICK_SMOKE for plumbing.

## CONFIG

In [ ]:
from pathlib import Path
from datetime import datetime
import glob

CONFIG = {
    "PROJECT_ROOT": Path.home() / "MAG2D-NC",
    "SEEDS": [0, 1, 2, 3, 4],
    "N_SPLITS": 5,
    "N_REPEATS": 3,
    "HP_TRIALS": 50,
    "INNER_SPLITS": 3,
    "PRIMARY": "f1_macro",
    "LC_FRACS": [0.25, 0.50, 0.75, 1.00],
    "RUN_STAMP": datetime.now().strftime("%Y%m%d-%H%M%S"),
}
QUICK_SMOKE = False
if QUICK_SMOKE:
    CONFIG.update({"SEEDS":[0], "N_REPEATS":1, "HP_TRIALS":5})
    print("*** SMOKE MODE — results not reportable ***")

feats = sorted(glob.glob(str(CONFIG["PROJECT_ROOT"]/"dataset"/"features_T1_*.parquet")))
CONFIG["FEATURES"] = Path(feats[-1])
OUT = CONFIG["PROJECT_ROOT"]/"output"; FIG = CONFIG["PROJECT_ROOT"]/"fig"
FIG.mkdir(exist_ok=True)
print(CONFIG["FEATURES"].name, "|", {k:v for k,v in CONFIG.items() if k not in ("PROJECT_ROOT","FEATURES")})

## Dependencies

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [("shap","shap"), ("lightgbm","lightgbm"), ("matplotlib","matplotlib")]:
    try:
        importlib.import_module(mod); print(pkg, "OK")
    except (ImportError, OSError):
        subprocess.run([sys.executable,"-m","pip","install",pkg], check=True)
        print(pkg, "installed")

## Load data

In [ ]:
import pandas as pd, numpy as np

df = pd.read_parquet(CONFIG["FEATURES"])
fcols = [c for c in df.columns if c.startswith(("comp_","soc_","sym_"))]
X_all = df[fcols].to_numpy(float)
y = (df["label2"]=="non_collinear").astype(int).to_numpy()
groups = df["group_id"].to_numpy()
label4 = df["label4"].to_numpy()
print(f"X {X_all.shape} | groups {df.group_id.nunique()} | pos {y.sum()} neg {(1-y).sum()}")

## Machinery: models, CV runner with artifact collection
One runner used by the main pass and A1; collects fold metrics, OOF predictions,
SHAP values (lgbm) and standardized coefficients (logreg), all mapped back to
full feature space through the VarianceThreshold support mask.

In [ ]:
import shap
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score
from scipy.stats import loguniform, randint, uniform
from collections import Counter
import warnings, time
warnings.filterwarnings("ignore", category=FutureWarning)

SPACES = {
  "logreg": {"clf__C": loguniform(1e-3, 1e3)},
  "lgbm":   {"clf__n_estimators": randint(100, 800),
             "clf__learning_rate": loguniform(0.005, 0.3),
             "clf__num_leaves": randint(4, 64),
             "clf__min_child_samples": randint(3, 30),
             "clf__subsample": uniform(0.6, 0.4),
             "clf__colsample_bytree": uniform(0.5, 0.5),
             "clf__reg_lambda": loguniform(1e-3, 10)}}

def make_pipe(name):
    if name == "logreg":
        return Pipeline([("vt",VarianceThreshold(0.0)),("sc",StandardScaler()),
                         ("clf",LogisticRegression(max_iter=5000, class_weight="balanced"))])
    return Pipeline([("vt",VarianceThreshold(0.0)),
                     ("clf",LGBMClassifier(objective="binary", verbosity=-1,
                                           class_weight="balanced", n_jobs=-1))])

def run_cv(Xsub, cols, tag, collect_artifacts=False):
    res = {m: {"folds": [], "oof": {}, "params": []} for m in ("logreg","lgbm")}
    if collect_artifacts:
        art = {"shap": np.zeros((len(CONFIG["SEEDS"])*CONFIG["N_REPEATS"], len(y), len(cols))),
               "coef": np.zeros_like(np.zeros((len(CONFIG["SEEDS"])*CONFIG["N_REPEATS"], len(cols)))),
               "coef_n": 0}
    ri = -1
    t0 = time.time()
    for seed in CONFIG["SEEDS"]:
        for rep in range(CONFIG["N_REPEATS"]):
            ri += 1
            rs = 1000*seed + rep
            outer = StratifiedGroupKFold(CONFIG["N_SPLITS"], shuffle=True, random_state=rs)
            folds = list(outer.split(Xsub, y, groups))
            for mname in ("logreg","lgbm"):
                oof_pred = np.full(len(y), -1)
                coef_acc = np.zeros(len(cols)); coef_k = 0
                for tr, te in folds:
                    inner = StratifiedGroupKFold(CONFIG["INNER_SPLITS"], shuffle=True, random_state=rs)
                    s = RandomizedSearchCV(make_pipe(mname), SPACES[mname],
                                           n_iter=CONFIG["HP_TRIALS"], scoring="f1_macro",
                                           cv=list(inner.split(Xsub[tr], y[tr], groups[tr])),
                                           random_state=rs, n_jobs=-1)
                    s.fit(Xsub[tr], y[tr])
                    best = s.best_estimator_
                    res[mname]["params"].append(s.best_params_)
                    yp = best.predict(Xsub[te]); oof_pred[te] = yp
                    res[mname]["folds"].append({
                        "tag": tag, "model": mname, "seed": seed, "rep": rep,
                        "f1_macro": f1_score(y[te], yp, average="macro"),
                        "mcc": matthews_corrcoef(y[te], yp),
                        "bal_acc": balanced_accuracy_score(y[te], yp)})
                    if collect_artifacts:
                        sup = best.named_steps["vt"].get_support()
                        if mname == "lgbm":
                            Xte_t = best.named_steps["vt"].transform(Xsub[te])
                            sv = shap.TreeExplainer(best.named_steps["clf"]).shap_values(Xte_t)
                            sv = sv[1] if isinstance(sv, list) else sv
                            art["shap"][ri][np.ix_(te, np.where(sup)[0])] = sv
                        else:
                            coef_acc[sup] += np.abs(best.named_steps["clf"].coef_.ravel()); coef_k += 1
                res[mname]["oof"][(seed,rep)] = oof_pred
                if collect_artifacts and mname == "logreg":
                    art["coef"][ri] = coef_acc/max(coef_k,1); art["coef_n"] += 1
            print(f"[{tag}] seed {seed} rep {rep} | {time.time()-t0:.0f}s")
    return (res, art) if collect_artifacts else (res, None)

print("machinery ready")

## Main pass (full features) with artifact collection

In [ ]:
res_full, art = run_cv(X_all, fcols, "full", collect_artifacts=True)
for m in ("logreg","lgbm"):
    v = np.array([f["f1_macro"] for f in res_full[m]["folds"]])
    print(f"{m}: f1_macro {v.mean():.3f} ± {v.std(ddof=1):.3f}  (n={len(v)})")

## Interpretability: rankings + cross-seed stability
Mean |SHAP| per feature (averaged over repeats and materials) and mean |coef|;
stability = mean pairwise Spearman of per-repeat rankings. Physics question on
the table: do `soc_*` and `sym_has_inversion` carry weight, and in which
direction?

In [ ]:
from scipy.stats import spearmanr
from itertools import combinations

shap_mean_per_run = np.abs(art["shap"]).mean(axis=1)          # runs x features
shap_imp = shap_mean_per_run.mean(axis=0)
coef_imp = art["coef"].mean(axis=0)

def stability(mat):
    rows = [r for r in mat if r.std() > 0]
    cors = [spearmanr(a, b).statistic for a, b in combinations(rows, 2)]
    return float(np.mean(cors))
print(f"rank stability (Spearman): shap={stability(shap_mean_per_run):.3f} "
      f"| coef={stability(art['coef']):.3f}")

imp = pd.DataFrame({"feature": fcols, "shap": shap_imp, "coef": coef_imp})
imp["shap_rank"] = imp["shap"].rank(ascending=False)
imp["coef_rank"] = imp["coef"].rank(ascending=False)
cross = spearmanr(imp["shap"], imp["coef"]).statistic
print(f"SHAP vs |coef| agreement (Spearman): {cross:.3f}")

print("\nTop-15 by SHAP:")
print(imp.sort_values("shap", ascending=False).head(15)[["feature","shap","coef_rank"]].to_string(index=False))
soc_sym = imp[imp.feature.str.startswith(("soc_","sym_"))].sort_values("shap", ascending=False)
print("\nAll soc_/sym_ features (SHAP rank of", len(fcols), "):")
print(soc_sym[["feature","shap_rank","coef_rank"]].to_string(index=False))
imp.to_csv(OUT/f"T1_importances_{CONFIG['RUN_STAMP']}.csv", index=False)

## Direction of effect for key physics features
Signed SHAP dependence: mean SHAP value in low vs high halves of each feature.
Positive = pushes toward non-collinear.

In [ ]:
KEY = ["sym_has_inversion","soc_anion_Z_wmean","soc_anion_Z_max",
       "soc_heavy_anion_frac","sym_mag_species_Z"]
shap_all = art["shap"].mean(axis=0)   # materials x features (averaged over runs)
for k in KEY:
    j = fcols.index(k)
    x = X_all[:, j]
    lo, hi = x <= np.median(x), x > np.median(x)
    print(f"{k:24s} mean SHAP low={shap_all[lo,j].mean():+.4f} "
          f"high={shap_all[hi,j].mean():+.4f}")

## Confusion analysis (pooled OOF) + fine-class resolution

In [ ]:
from sklearn.metrics import confusion_matrix

for mname in ("logreg","lgbm"):
    preds = np.concatenate([p for p in res_full[mname]["oof"].values()])
    truth = np.tile(y, len(res_full[mname]["oof"]))
    cm = confusion_matrix(truth, preds)
    print(f"\n{mname} pooled OOF confusion [rows=true col,nc]:\n{cm}")
    fine = np.tile(label4, len(res_full[mname]["oof"]))
    tab = pd.crosstab(pd.Series(fine, name="true_label4"),
                      pd.Series(preds, name="pred_binary"))
    tab["recall_as_own_class"] = tab.apply(
        lambda r: r[1]/(r[0]+r[1]) if r.name in ("NC","DM_SS") else r[0]/(r[0]+r[1]), axis=1)
    print(tab.round(3).to_string())

## Ablation A1 — descriptor-group removal (full nested search)

In [ ]:
abl_rows = []
for drop in ("comp","soc","sym"):
    keep_cols = [c for c in fcols if not c.startswith(drop+"_")]
    Xs = df[keep_cols].to_numpy(float)
    res_a, _ = run_cv(Xs, keep_cols, f"-{drop}")
    for m in ("logreg","lgbm"):
        v = np.array([f["f1_macro"] for f in res_a[m]["folds"]])
        abl_rows.append({"variant": f"-{drop}", "model": m,
                         "f1_macro": v.mean(), "std": v.std(ddof=1)})
for m in ("logreg","lgbm"):
    v = np.array([f["f1_macro"] for f in res_full[m]["folds"]])
    abl_rows.append({"variant": "full", "model": m, "f1_macro": v.mean(), "std": v.std(ddof=1)})
abl = pd.DataFrame(abl_rows).pivot(index="variant", columns="model", values="f1_macro").round(4)
print(abl.to_string())
pd.DataFrame(abl_rows).to_csv(OUT/f"T1_ablation_A1_{CONFIG['RUN_STAMP']}.csv", index=False)

## Ablation A3 — learning curve (group-aware subsampling, modal HPs)
HPs fixed to the modal best of the main pass (disclosed); training GROUPS are
subsampled at each fraction so leakage control is preserved.

In [ ]:
def modal(plist):
    ks = plist[0].keys()
    return {k: Counter(p[k] for p in plist).most_common(1)[0][0] for k in ks}

lc_rows = []
for mname in ("logreg","lgbm"):
    pipe = make_pipe(mname).set_params(**modal(res_full[mname]["params"]))
    for frac in CONFIG["LC_FRACS"]:
        scores = []
        for seed in CONFIG["SEEDS"]:
            rs = 7000 + seed
            outer = StratifiedGroupKFold(CONFIG["N_SPLITS"], shuffle=True, random_state=rs)
            rng = np.random.default_rng(rs)
            for tr, te in outer.split(X_all, y, groups):
                gtr = np.unique(groups[tr])
                keep_g = rng.choice(gtr, size=max(2, int(round(frac*len(gtr)))), replace=False)
                tr_sub = tr[np.isin(groups[tr], keep_g)]
                if len(np.unique(y[tr_sub])) < 2:
                    continue
                pipe.fit(X_all[tr_sub], y[tr_sub])
                scores.append(f1_score(y[te], pipe.predict(X_all[te]), average="macro"))
        lc_rows.append({"model": mname, "frac": frac,
                        "f1_macro": float(np.mean(scores)), "std": float(np.std(scores, ddof=1)),
                        "n_folds": len(scores)})
        print(lc_rows[-1])
lc = pd.DataFrame(lc_rows)
lc.to_csv(OUT/f"T1_learning_curve_{CONFIG['RUN_STAMP']}.csv", index=False)

## Figures (paper drafts)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6,4.5))
top = imp.sort_values("shap", ascending=True).tail(15)
colors = ["tab:red" if f.startswith(("soc_","sym_")) else "tab:gray" for f in top.feature]
ax.barh(top.feature, top.shap, color=colors)
ax.set_xlabel("mean |SHAP| (OOF)"); ax.set_title("T1 top-15 descriptors (red = physics-informed)")
fig.tight_layout(); fig.savefig(FIG/f"T1_shap_top15_{CONFIG['RUN_STAMP']}.png", dpi=300)

fig2, ax2 = plt.subplots(figsize=(5,3.8))
for mname, mk in (("logreg","o-"), ("lgbm","s--")):
    sub = lc[lc.model==mname]
    ax2.errorbar(sub.frac*100, sub.f1_macro, yerr=sub["std"], fmt=mk, capsize=3, label=mname)
ax2.set_xlabel("% of training groups"); ax2.set_ylabel("F1-macro"); ax2.legend()
ax2.set_title("T1 learning curve (modal HPs)")
fig2.tight_layout(); fig2.savefig(FIG/f"T1_learning_curve_{CONFIG['RUN_STAMP']}.png", dpi=300)
print("figures saved to fig/")

## Log + environment

In [ ]:
import json as _j, subprocess, sys
folds_all = pd.DataFrame([f for m in ("logreg","lgbm") for f in res_full[m]["folds"]])
folds_all["run_stamp"] = CONFIG["RUN_STAMP"]; folds_all["task"] = "T1-interp"
runs_path = OUT/"runs.csv"
folds_all.to_csv(runs_path, mode="a", header=not runs_path.exists(), index=False)
(OUT/f"environment_T1interp_{CONFIG['RUN_STAMP']}.txt").write_text(
    subprocess.run([sys.executable,"-m","pip","freeze"],capture_output=True,text=True).stdout)
print("logged", len(folds_all), "fold rows | environment saved")
print("\nPaste back: stability lines, top-15 table, soc/sym table, direction table, "
      "confusion tables, A1 pivot, learning-curve rows.")